# Outliers

## Método IQR (Rango Intercuartil)
- Detectar outliers basado en el rango intercuartil. Robusto contra extremos.

In [156]:
def percentile(items: list, p: int) -> int:
    """get percentile """
    items_sorted = sorted(items)
    num = len(items_sorted) * p
    deno = 100
    index = int(num / deno)
    return items_sorted[index]


def detect_outliers(items: list, key: str) -> list:
    """detect outliers by key"""

    values = [i[key] for i in items]
    print(f"values {values}")

    p_25 = percentile(values, 25)
    p_75 = percentile(values, 75)
    iqr = p_75 - p_25

    lower_limit = p_25 - (1.5 * iqr)
    upper_limit = p_75 + (1.5 * iqr)

    outliers = []
    normal = []
    
    for idx, i in enumerate(items):
        current_value = i[key]
        if current_value < lower_limit or current_value > upper_limit:
            outliers.append((idx, i))
        else:
            normal.append(i)
    
    return {
        "p_25": p_25,
        "p_75": p_75,
        "iqr": iqr,
        "lower_limit": lower_limit,
        "upper_limit": upper_limit,
        "outliers": outliers,
        "normal": normal
    }


# example
datos = [
    {"id": 1, "edad": 25},
    {"id": 2, "edad": 28},
    {"id": 3, "edad": 30},
    {"id": 4, "edad": 32},
    {"id": 5, "edad": 35},
    {"id": 6, "edad": 45},
    {"id": 7, "edad": 120}  # Outlier
]


outliers = detect_outliers(items=datos, key="edad")
print("\nOutliers")
print(f"\tp_25: {outliers["p_25"]}")
print(f"\tp_75: {outliers["p_75"]}")
print(f"\tiqr: {outliers["iqr"]}")
print(f"\tlower_limit: {outliers["lower_limit"]}")
print(f"\tupper_limit: {outliers["upper_limit"]}")
print(f"\toutliers: {outliers["outliers"]}")
print(f"\normal: {outliers["normal"]}")

values [25, 28, 30, 32, 35, 45, 120]

Outliers
	p_25: 28
	p_75: 45
	iqr: 17
	lower_limit: 2.5
	upper_limit: 70.5
	outliers: [(6, {'id': 7, 'edad': 120})]

ormal: [{'id': 1, 'edad': 25}, {'id': 2, 'edad': 28}, {'id': 3, 'edad': 30}, {'id': 4, 'edad': 32}, {'id': 5, 'edad': 35}, {'id': 6, 'edad': 45}]


# Z-Score para Outliers

- Un valor es outlier si su Z-score es > 3 (muy lejos de la media).

In [157]:
from statistics import mean, stdev
import math


def get_outliers_z_score(items: list, key: str, umbral=3) -> tuple:
    """using z score"""

    values = [i[key] for i in items]
    values_mean = mean(values)
    values_des = stdev(values) if len(values) > 0 else 0

    if values_des == 0:
        return ([], mean, 0)

    outliers = []
    for i, r in enumerate(items):
        v = r[key]
        z_score = (v-values_mean) / values_des
        # print(f"id: {r["id"]} | z: {z_score}")
        if abs(z_score) > umbral:
            outliers.append((i, v, r, z_score))

    return (outliers, values_mean, values_des)


# EJEMPLO
datos = [
    {"id": 1, "salario": 40000},
    {"id": 2, "salario": 45000},
    {"id": 3, "salario": 48000},
    {"id": 4, "salario": 50000},
    {"id": 5, "salario": 52000},
    {"id": 6, "salario": 55000},
    {"id": 7, "salario": 1000000},  # Outlier extremo
]

print("\nOutliers using z-score")
outliers_z_score = get_outliers_z_score(datos, "salario", 2)
print(f"\tOutliers:")
for i in outliers_z_score[0]:
    print(f"\t\tid: {i[0]} | value: {i[1]} | z_score: {i[3]}")
print(f"\tMean: {outliers_z_score[1]:.4f}")
print(f"\tDesv: {outliers_z_score[2]:.4f}")


Outliers using z-score
	Outliers:
		id: 6 | value: 1000000 | z_score: 2.267580426211339
	Mean: 184285.7143
	Desv: 359728.9323


## Tipos de Outliers: Verdaderos vs Errores
- No todos los outliers son errores. Necesitas investigar y decidir.

In [158]:
# TIPOS DE OUTLIERS

# 1. ERROR DE ENTRADA (debe eliminarse)
datos_errores = [
    {"edad": 25},
    {"edad": 30},
    {"edad": 999},  # Probablemente '99' mal digitado
    {"edad": -5},   # Imposible
    {"edad": 35}
]

print("1. ERRORES DE ENTRADA (eliminar):")
print("   Edad: 999 (máximo humano ~122)")
print("   Edad: -5 (imposible)")

# 2. VARIABILIDAD NATURAL (mantener)
datos_variables = [
    {"ciudad": "Madrid", "población": 3000000},
    {"ciudad": "Barcelona", "población": 1500000},
    {"ciudad": "Pueblito", "población": 500},
]

print("\n2. VARIABILIDAD NATURAL (mantener):")
print("   Pueblito vs Madrid es esperado en datos de ciudades")

# 3. EVENTO INUSUAL (decidir según contexto)
datos_eventos = [
    {"empresa": "TechCorp", "ingresos_anuales": 5000000},
    {"empresa": "TechCorp", "ingresos_q3": 2000000},
    {"empresa": "TechCorp", "ingresos_q4": 8000000},  # Q4 típicamente mayor
]

print("\n3. EVENTOS INUSUALES (investigar):")
print("   Q4 mayor que otros trimestres es real, no error")

# FUNCIÓN PARA CLASIFICAR
def clasificar_outlier(valor, campo, contexto):
    """Clasificar si un outlier es error o válido"""
    
    reglas = {
        "edad": {"min": 0, "max": 125},
        "salario": {"min": 0},
        "temperatura": {"min": -50, "max": 60},
        "cantidad": {"min": 0}
    }
    
    if campo not in reglas:
        return "desconocido"
    
    regla = reglas[campo]
    
    if "min" in regla and valor < regla["min"]:
        return "error_invalido"
    if "max" in regla and valor > regla["max"]:
        return "error_invalido"
    
    return "valido"

print("\nClasificación:")
pruebas = [
    ("edad", 25, "normal"),
    ("edad", 999, "error"),
    ("salario", 1000000, "extremo_pero_válido"),
    ("temperatura", -100, "error")
]

for campo, valor, contexto in pruebas:
    clasificacion = clasificar_outlier(valor, campo, contexto)
    print(f"  {campo}={valor}: {clasificacion}")
      

1. ERRORES DE ENTRADA (eliminar):
   Edad: 999 (máximo humano ~122)
   Edad: -5 (imposible)

2. VARIABILIDAD NATURAL (mantener):
   Pueblito vs Madrid es esperado en datos de ciudades

3. EVENTOS INUSUALES (investigar):
   Q4 mayor que otros trimestres es real, no error

Clasificación:
  edad=25: valido
  edad=999: error_invalido
  salario=1000000: valido
  temperatura=-100: error_invalido


## Estrategias de Manejo de Outliers

> Opciones para tratar outliers: eliminar, transformar, aislar o imputar.

### ESTRATEGIA 1: ELIMINAR OUTLIERS

In [159]:
from statistics import mean

# ESTRATEGIA 1: ELIMINAR OUTLIERS

def drop_outliers(items: list, key: str):
    """delete by outliers"""

    values = [r[key] for r in items]

    q1 = sorted(values)[len(values)//4]
    q2 = sorted(values)[3*len(values)//4]
    iqr = q2-q1

    lower_limit = q1 - (1.5 * iqr)
    upper_limit = q2 + (1.5 * iqr)

    result = []
    for i in items:
        if lower_limit <= i[key] <= upper_limit:
            result.append(i)

    return result


datos = [
    {"ingresos": 40000},
    {"ingresos": 50000},
    {"ingresos": 48000},
    {"ingresos": 52000},
    {"ingresos": 5000000}  # Outlier
]

datos_filtered = drop_outliers(datos, "ingresos")
print(f"\nDelete outliers")
print(f"\tSize datos: {len(datos)}")
print(f"\tSize filteres: {len(datos_filtered)}")
print(f"\tDeleted: {len(datos) - len(datos_filtered)}")


Delete outliers
	Size datos: 5
	Size filteres: 3
	Deleted: 2


### ESTRATEGIA 2: CAPEAR (Winsorize) - Limitar al percentil

In [160]:
def winsorize(items: list, key: str, percentile=5) -> list:
    """implement winsorize"""

    vals = sorted([r[key] for r in items])
    
    print(f"vals: {vals} | len: {len(vals)} | {len(vals) * percentile} | {len(vals) * (100 - percentile)} ")

    p1_raw = (len(vals) * percentile) / 100
    p2_raw = (len(vals) * (100 - percentile)) / 100
    print(f"{p1_raw} -> {int(p1_raw)} | {p2_raw} -> {int(p2_raw)}")
    
    lower_limit = vals[int(p1_raw)]
    upper_limit = vals[int(p2_raw)]
    
    print(f"percentile: {percentile} | lower_limit: {lower_limit} | upper_limit: {upper_limit}")

    result = []
    for i in items:
        i_copy = i.copy()
        c_val = i[key]
        
        print(f"c_val: {c_val}")

        if c_val < lower_limit:
            i_copy[key] = lower_limit

        if c_val > upper_limit:
            i_copy[key] = upper_limit

        result.append(i_copy)

    return result


datos = [
    {"ingresos": 40000},
    {"ingresos": 50000},
    {"ingresos": 48000},
    {"ingresos": 52000},
    {"ingresos": 5000000}  # Outlier
]


datos_by_winsorize = winsorize(datos, "ingresos", percentile=25)
for i, j in zip(datos, datos_by_winsorize):
    print(f"i: {i["ingresos"]} -> {j["ingresos"]}")

vals: [40000, 48000, 50000, 52000, 5000000] | len: 5 | 125 | 375 
1.25 -> 1 | 3.75 -> 3
percentile: 25 | lower_limit: 48000 | upper_limit: 52000
c_val: 40000
c_val: 50000
c_val: 48000
c_val: 52000
c_val: 5000000
i: 40000 -> 48000
i: 50000 -> 50000
i: 48000 -> 48000
i: 52000 -> 52000
i: 5000000 -> 52000


### ESTRATEGIA 3: TRANSFORMACIÓN LOGARÍTMICA

In [161]:
def transformation_log(items: list, key: str) -> list: 
    """using log"""
    
    result = []
    for i in items:
        i_copy = i.copy()
        value = i_copy[key]
        if value > 0:
            i_copy[f"{key}_log"] = math.log10(value)
        else:
            i_copy[f"{key}_log"] = 0
        result.append(i_copy)
    
    return result


datos = [
    {"ingresos": 40000},
    {"ingresos": 50000},
    {"ingresos": 48000},
    {"ingresos": 52000},
    {"ingresos": 5000000}  # Outlier
]

datos_log = transformation_log(datos, "ingresos")
for i in datos_log:
    print(f"{i["ingresos"]} -> {i["ingresos_log"]}")

40000 -> 4.6020599913279625
50000 -> 4.698970004336019
48000 -> 4.681241237375588
52000 -> 4.7160033436347994
5000000 -> 6.698970004336019


### ESTRATEGIA 4: IMPUTAR CON MEDIA TRUNCADA (sin extremos)

In [165]:
def set_mean(items: list, key: str, percentil=10):
    """Imputar by mean"""

    items_sorted = sorted(i[key] for i in items)

    # set the range for percentil
    p_1 = int(len(items_sorted) * percentil / 100)
    p_2 = int(len(items_sorted) * (100 - percentil) / 100)
    items_ranged = items_sorted[p_1:p_2]
    items_ranged_mean = mean(items_ranged)
    
    print(f"items_ranged_mean: {items_ranged_mean} -> {items_ranged}")

    # computed percentil
    result = []
    q_1 = sorted(items_sorted)[len(items_sorted) // 4]
    q_2 = sorted(items_sorted)[3 * (len(items_sorted) // 4)]
    iqr = q_2 - q_1
    lower_limit = q_1 - 1.5 * iqr
    upper_limit = q_2 + 1.5 * iqr
    
    for r in items:
        r_copy = r.copy()
        if r_copy[key] < lower_limit or r_copy[key] > upper_limit:
            r_copy[key] = items_ranged_mean
        result.append(r_copy)
        
    return result


datos = [
    {"ingresos": 40000},
    {"ingresos": 50000},
    {"ingresos": 48000},
    {"ingresos": 52000},
    {"ingresos": 5000000}  # Outlier
]

setted_datos_mean = set_mean(datos, "ingresos", percentil=10)
for i,j in zip(datos, setted_datos_mean):
    print(f"{i["ingresos"]} | {j["ingresos"]}")

items_ranged_mean: 47500 -> [40000, 48000, 50000, 52000]
40000 | 47500
50000 | 50000
48000 | 48000
52000 | 52000
5000000 | 47500


## Tips y Mejores Prácticas

| Icon | Recomendación |
|------|----------------|
| 💡 | Usar IQR es más robusto que Z-score. IQR no asume distribución normal. |
| ⚠️ | Antes de eliminar un outlier, investiga. ¿Es error o dato válido inusual? |
| 💡 | Winsorize (capear) es mejor que eliminar si hay pocos outliers extremos. |
| ℹ️ | En dominio conocido, establece límites realistas (ej: edad 0-150) y rechaza lo imposible. |
| ⚠️ | Eliminar muchos outliers (>5%) distorsiona el análisis. Cuestiona la calidad de datos. |
| 💡 | Los outliers PODRÍAN ser el patrón más interesante. En fraude, los outliers son lo importante. |

---

## Errores Comunes

### 1. Eliminar todos los outliers sin investigar

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | El mejor cliente (máximo gasto) es un outlier. Los fraudes son outliers. Son datos valiosos. |
| **Solución** | Investiga cada outlier. Clasifica como error vs válido. Elimina solo errores. |

---

### 2. Usar Z-score cuando hay distribución sesgada

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Z-score asume distribución normal. Datos sesgados lo falsean. |
| **Solución** | Usa IQR para datos reales (más robusto) o verifica distribución primero. |

---

### 3. Comparar outliers de campos con diferentes escalas

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Un Z-score de 2 en edad es raro. Un Z-score de 2 en ingresos es normal. |
| **Solución** | Normaliza ANTES de detectar, o usa métodos por campo independiente. |

---

### 4. Eliminar outliers y luego reportar estadísticas con "n original"

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Engañoso. Las personas usan n original pero datos sin outliers. |
| **Solución** | Reporta: "n=100 original, n=95 después de limpiar outliers". |

---

### 5. Asumir que "outlier" siempre significa "incorrecto"

| Aspecto | Detalle |
|---------|----------|
| **¿Por qué ocurre?** | Outlier = inusual. Incorrecto = error. No son lo mismo. |
| **Solución** | Outlier correcto: CEO con salario alto. Outlier incorrecto: edad = 999. |